In [ ]:
import cv2
import os
import mediapipe as mp
import numpy as np
# import cv_bridge
# import rosbags
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import moviepy as mpy

In [ ]:
pose = ['nose',
        'l_eye_inner','l_eye','l_eye_outer',
        'r_eye_inner','r_eye','r_eye_outer',
        'l_ear','r_ear',
        'l_mouth','r_mouth',
        'l_shoulder','r_shoulder',
        'l_elbow','r_elbow',
        'l_wrist','r_wrist',
        'l_pinky','r_pinky',
        'l_index','r_index',
        'l_thumb','r_thumb',
        'l_hip','r_hip',
        'l_knee','r_knee',
        'l_ankle','r_ankle',
        'l_heel','r_heel',
        'l_foot_index','r_foot_index'
        ]
coord = ['x','y','z','visability']

combinations = ["_".join(x) for x in itertools.product(pose, coord)]
combinations.extend(['frame_no'])

# using timestamps to order the list of images in sequence
def get_ts(i):
    return(float(i[-32:-4]))

def pngToVid(
        image_folder,video_name, 
        sec_front_trim = np.nan, sec_end_trim = np.nan, # trim based on seconds in video
        start_ts = np.datetime64('NaT'), # trim based on specific ts to align with other videos
        end_ts = np.datetime64('NaT'),
        reSize = False):
     images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
     print(len(images))
     frame = cv2.imread(os.path.join(image_folder, images[0]))
     height, width, layers = frame.shape
     if reSize:
        width = width//3 # will crop to middle third on x axis
     size = (width,height)
     if not (np.isnan(sec_end_trim) & np.isnan(sec_front_trim)):
        min_ts = get_ts(min(images, key=get_ts))
        min_ts = pd.to_datetime(str(min_ts), unit='ms')
        # sec_front_trim
        start = min_ts + pd.Timedelta(unit='second',value=sec_front_trim)
        stop = min_ts + pd.Timedelta(unit='second',value=sec_end_trim)
        print(min_ts)
        images = [img for img in images if (pd.to_datetime(str(get_ts(img)), unit='ms') >= start)]
        images = [img for img in images if (pd.to_datetime(str(get_ts(img)), unit='ms') <= stop)]
        print(len(images))
    #  elif (start_ts != np.datetime64('NaT')) & (end_ts != np.datetime64('NaT')):
     elif not(pd.isnull(start_ts) & pd.isnull(end_ts)):
        print('here...')
        print(start_ts)
        print(end_ts)
        images = [img for img in images if (pd.to_datetime(str(get_ts(img)), unit='ms') >= start_ts)]
        images = [img for img in images if (pd.to_datetime(str(get_ts(img)), unit='ms') <= end_ts)]
        print(len(images))
     images.sort(key = get_ts)
     # combine images into video
     video = cv2.VideoWriter(
         filename = video_name, 
         fourcc = cv2.VideoWriter_fourcc(*'MP4V'),
         fps = 30, frameSize = size)
     
     for image in images:
         img = cv2.imread(os.path.join(image_folder, image))
         # this handles cropping if necessary
         if reSize:
             img = img[0:height,width:(width*2)]
         video.write(img)
     cv2.destroyAllWindows()
     video.release()

     # get timestamps to align with landmark and angle measures

     time_stamps = [str(get_ts(image)) for image in images]
     ts_df = pd.DataFrame(
         columns=['timeStamp'],
         data=time_stamps
         )
     ts_df['timeStamp'] = pd.to_datetime(ts_df.timeStamp, unit='ms')
     ts_df['frame_no'] = list(range(0, len(time_stamps)))
     len(ts_df)

     return(ts_df)

def getAngle(a,b,c):

    a = np.array(a) # angle for this joint
    b = np.array(b)
    c = np.array(c)

    ab = a - b
    ac = a - c
    cosine_angle = np.dot(ab,ac) / (np.linalg.norm(ab) * np.linalg.norm(ac))
    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

def doSkeletalTracking(video_name,ts_df,blackOut):
    #https://github.com/google-ai-edge/mediapipe/blob/master/docs/solutions/pose.md
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles
    mp_pose = mp.solutions.pose
    print(video_name)
    cap = cv2.VideoCapture(video_name)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    size = (width,height)
    # print(video_name[-4:]+'_pose'+video_name[-4:])
    video = cv2.VideoWriter(
        filename = video_name[:-4]+'_pose'+video_name[-4:], 
        fourcc = cv2.VideoWriter_fourcc(*'MP4V'),
        fps = 30, frameSize = size)
    landmarks_data = []
    world_landmarks_data = []

    with mp_pose.Pose(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as pose:
        while cap.isOpened():
            success, image = cap.read()
            print(success)
            if not success:
                print("Ignoring empty camera frame.")
                # If loading a video, use 'break' instead of 'continue'.
                # # continue
                break
             # To improve performance, optionally mark the image as not writeable to
             # pass by reference.
            image.flags.writeable = False
            # if blackOut:
            #     image = np.zeros((height,width,3), dtype=np.uint8)
            # else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = pose.process(image)
            # Extract landmarks and save to dataframe
            if results.pose_landmarks:
                frame_landmarks = []
                for landmark in results.pose_landmarks.landmark:
                    frame_landmarks.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
                frame_landmarks.extend([int(cap.get(cv2.CAP_PROP_POS_FRAMES))-1])
                landmarks_data.append(frame_landmarks)
                
                frame_world_landmarks = []
                for landmark in results.pose_world_landmarks.landmark:
                    frame_world_landmarks.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
                frame_world_landmarks.extend([int(cap.get(cv2.CAP_PROP_POS_FRAMES))-1])
                world_landmarks_data.append(frame_world_landmarks)
            # Draw the pose annotation on the image.
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())
            # Flip the image horizontally for a selfie-view display.
            # cv2.imshow('MediaPipe Pose', cv2.flip(image, 1))
            video.write(image)
            if cv2.waitKey(5) & 0xFF == 27:
                break
    print(len(landmarks_data))
    # joins with timestamp info
    land_marks_df = pd.merge(
        pd.DataFrame(landmarks_data, columns = combinations),
        ts_df, 
        on = 'frame_no',how='left')
    land_marks_df.to_csv(video_name+'_landmark.csv', index=False)
    w_land_marks_df = pd.merge(
        pd.DataFrame(world_landmarks_data, columns = combinations),
        ts_df,
        on = 'frame_no',how='left')
    w_land_marks_df.to_csv(video_name+'_world_landmark.csv', index=False)
    cap.release()
    cv2.destroyAllWindows()
    video.release()

    return land_marks_df, w_land_marks_df
    
jointPoints = {
    'shoulder': {'a':'shoulder','b':'hip','c':'elbow'},
    'elbow': {'a':'elbow','b':'wrist','c':'shoulder'},
    'hip': {'a':'hip','b':'shoulder','c':'knee'},
    'knee': {'a':'knee','b':'hip','c':'ankle'}
    }

def getJointAngles(df,reSampUnit,jointPoints):
    sides = ['l','r']
    joints = ['shoulder','elbow','hip','knee']
    for side in sides:
        print(side)
        for joint in joints:
            var = str(side+"_"+joint+"_angle")
            df[var] = df.apply(lambda x: getAngle(
                a = [x[str(side+"_"+jointPoints[joint]['a']+"_x")],
                     x[str(side+"_"+jointPoints[joint]['a']+"_y")],
                     x[str(side+"_"+jointPoints[joint]['a']+"_z")]], 
                b = [x[str(side+"_"+jointPoints[joint]['b']+"_x")],
                     x[str(side+"_"+jointPoints[joint]['b']+"_y")],
                     x[str(side+"_"+jointPoints[joint]['b']+"_z")]], 
                c = [x[str(side+"_"+jointPoints[joint]['c']+"_x")],
                     x[str(side+"_"+jointPoints[joint]['c']+"_y")],
                     x[str(side+"_"+jointPoints[joint]['c']+"_z")]]), 
                axis = 1
                )
    df = df.set_index('timeStamp').resample(reSampUnit).mean()
    return(df)

def getAnglePlot(df,view,part_no,project_dir):

    if view == 'FV':
        angles = ['l_elbow_angle','r_elbow_angle','l_shoulder_angle','r_shoulder_angle']
    else:
        angles = ['l_shoulder_angle','l_hip_angle','l_knee_angle']
    df.plot.line(
        y = angles
        )
    plt.ylim(0,180)
    plt.title(view+"_"+part_no)
    plt.savefig(os.path.join(project_dir,view,part_no,view+'_'+part_no+'.png'))
    return plt

def combineVideos(FV_vid,SV_vid,FV_plt,SV_plt,sav_fname):
    videoFV = mpy.VideoFileClip(FV_vid)
    chartFV = mpy.ImageClip(FV_plt) 
    chartFV.duration = videoFV.duration
    
    videoSV = mpy.VideoFileClip(SV_vid)
    chartSV = mpy.ImageClip(SV_plt)
    chartSV.duration = videoSV.duration
    
    combined = mpy.clips_array(
        [[videoSV,chartSV],
         [videoFV,chartFV]]
         )
    # combined.preview(fps=30)
    combined.write_videofile(sav_fname)

In [ ]:
project_dir = '/Volumes/LaCie/RealSense/CPR_test_data_02-27-2025'
# part_nos = ['P1','P2','P3','P4','P5']
part_no = 'P5'

In [ ]:
# FV_png_fldr = '/Volumes/LaCie/RealSense/CPR_test_data_02-27-2025/FV/P3'
# SV_png_fldr = '/Volumes/LaCie/RealSense/CPR_test_data_02-27-2025/SV/P3'
# FV_vid_name = 'FV_3.mp4'
# SV_vid_name = 'SV_3.mp4'

# FV_1_ts_df = pngToVid(
#     image_folder = FV_png_fldr,
#     reSize = True,
#     video_name = FV_vid_name,
#     sec_front_trim = 5,
#     sec_end_trim = 124 # the second in video duration to stop the clip (not # of seconds to cut off end)
# )
# print(min(FV_1_ts_df.timeStamp))
# print(max(FV_1_ts_df.timeStamp))



In [ ]:
FV_ts_df = pngToVid(
    image_folder = os.path.join(project_dir,'FV',part_no,'raw_pngs'),
    reSize = True,
    video_name = os.path.join(project_dir,'FV',part_no,'FV_'+part_no+'.mp4')#,
    # sec_front_trim = 97,
    # sec_end_trim = 215 # the second in video duration to stop the clip (not # of seconds to cut off end)
)

SV_ts_df = pngToVid(
    image_folder = os.path.join(project_dir,'SV',part_no,'raw_pngs'),
    video_name = os.path.join(project_dir,'SV',part_no,'SV_'+part_no+'.mp4'),
    reSize=True#,
    # sec_front_trim=105,
    # sec_end_trim=223
    # start_ts=min(FV_1_ts_df.timeStamp),
    # end_ts=max(FV_1_ts_df.timeStamp)
)

In [ ]:
blackOut = False
FV_ts_df = pngToVid(
    image_folder = os.path.join(project_dir,'FV',part_no,'raw_pngs'),
    reSize = True,
    video_name = os.path.join(project_dir,'FV',part_no,'FV_'+part_no+'.mp4'),
    sec_front_trim = 24,
    sec_end_trim = 139 # the second in video duration to stop the clip (not # of seconds to cut off end)
)

SV_ts_df = pngToVid(
    image_folder = os.path.join(project_dir,'SV',part_no,'raw_pngs'),
    video_name = os.path.join(project_dir,'SV',part_no,'SV_'+part_no+'.mp4'),
    reSize=True,
    sec_front_trim=30,
    sec_end_trim=146
    # start_ts=min(FV_1_ts_df.timeStamp),
    # end_ts=max(FV_1_ts_df.timeStamp)
)

FV_LM_df, FV_wLM_df= doSkeletalTracking(
    video_name = os.path.join(project_dir,'FV',part_no,'FV_'+part_no+'.mp4'),
    ts_df = FV_ts_df,
    blackOut = blackOut
)

SV_LM_df, SV_wLM_df= doSkeletalTracking(
    video_name = os.path.join(project_dir,'SV',part_no,'SV_'+part_no+'.mp4'),
    ts_df = SV_ts_df,
    blackOut = blackOut
)

FV_df = getJointAngles(
    df=FV_LM_df,
    reSampUnit='50ms',
    jointPoints=jointPoints
)

SV_df = getJointAngles(
    df=SV_LM_df,
    reSampUnit='50ms',
    jointPoints=jointPoints
)

FV_plt = getAnglePlot(
    df = FV_df,
    view = 'FV',
    part_no = part_no,
    project_dir = project_dir)

SV_plt = getAnglePlot(
    df = SV_df,
    view = 'SV',
    part_no = part_no,
    project_dir = project_dir)

combineVideos(
    FV_vid = os.path.join(project_dir,'FV',part_no,'FV_'+part_no+'_pose.mp4'),
    SV_vid = os.path.join(project_dir,'SV',part_no,'SV_'+part_no+'_pose.mp4'),
    FV_plt = os.path.join(project_dir,'FV',part_no,"FV"+'_'+part_no+'.png'),
    SV_plt = os.path.join(project_dir,'SV',part_no,"SV"+'_'+part_no+'.png'),
    sav_fname = os.path.join(project_dir,"cmbd_vid_"+str(blackOut)+"_"+part_no+".mp4") 
    )

In [ ]:
x = os.path.join(project_dir,'SV',part_no,'SV_'+part_no+'.mp4')
x[:-4]

In [ ]:
height = 512
width = 512
channels = 3

# Create a black image using NumPy. All pixel values are initialized to 0.
black_image = np.zeros((height, width, channels), dtype=np.uint8)

# Display the black image (optional)
cv2.imshow("Black Image", black_image)

In [ ]:
# OLD read in png files and create video
image_folder = '/Volumes/LaCie/RealSense/12-05-2024/side_view/png_1'
video_name = 'SV_1.mp4'

images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
frame = cv2.imread(os.path.join(image_folder, images[0]))
height, width, layers = frame.shape
size = (width,height)

# using timestamps to order the list of images in sequence
def get_ts(i):
    return(float(i[-32:-4]))
images.sort(key = get_ts)
# combine images into video
video = cv2.VideoWriter(
    filename = video_name, 
    fourcc = cv2.VideoWriter_fourcc(*'MP4V'),
    fps = 30, frameSize = size)

for image in images:
    video.write(cv2.imread(os.path.join(image_folder, image)))

cv2.destroyAllWindows()
video.release()

In [ ]:
# OLD get timestamps to align with landmark and angle measures
time_stamps = [str(get_ts(image)) for image in images]
ts_df = pd.DataFrame(
    columns=['timeStamp'],
     data=time_stamps
)
ts_df['timeStamp'] = pd.to_datetime(ts_df.timeStamp, unit='ms')

ts_df['frame_no'] = list(range(0, len(time_stamps)))
len(ts_df)


In [ ]:
# OLD Do skeletal tracking, save video and landmarks
#https://github.com/google-ai-edge/mediapipe/blob/master/docs/solutions/pose.md
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_pose = mp.solutions.pose
video_name = 'SV_1.mp4'
cap = cv2.VideoCapture(video_name)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
size = (width,height)
video = cv2.VideoWriter(
  filename = "pose_"+video_name, 
  fourcc = cv2.VideoWriter_fourcc(*'MP4V'),
  fps = 30, frameSize = size)

landmarks_data = []
world_landmarks_data = []

with mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as pose:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      # continue
      break

    # To improve performance, optionally mark the image as not writeable to
    # pass by reference.
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image)
    # Extract landmarks and save to dataframe
    if results.pose_landmarks:
        frame_landmarks = []
        for landmark in results.pose_landmarks.landmark:
            frame_landmarks.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
        frame_landmarks.extend([int(cap.get(cv2.CAP_PROP_POS_FRAMES))-1])
        landmarks_data.append(frame_landmarks)

        frame_world_landmarks = []
        for landmark in results.pose_world_landmarks.landmark:
            frame_world_landmarks.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
        frame_world_landmarks.extend([int(cap.get(cv2.CAP_PROP_POS_FRAMES))-1])
        world_landmarks_data.append(frame_world_landmarks)
    # Draw the pose annotation on the image.
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())
    # Flip the image horizontally for a selfie-view display.
    # cv2.imshow('MediaPipe Pose', cv2.flip(image, 1))
    video.write(image)
    if cv2.waitKey(5) & 0xFF == 27:
      break
print(len(landmarks_data))
# joins with timestamp info
pd.merge(
   pd.DataFrame(landmarks_data, columns = combinations),
   ts_df, 
   on = 'frame_no',how='left').to_csv(video_name+'_landmark.csv', index=False)
pd.merge(
   pd.DataFrame(world_landmarks_data, columns = combinations),
   ts_df,
   on = 'frame_no',how='left').to_csv(video_name+'_world_landmark.csv', index=False)
cap.release()
cv2.destroyAllWindows()
video.release()

In [ ]:
# OLD get Joint angles 
video_name = 'SV_1.mp4'
# df = pd.read_csv(video_name+'_world_landmark.csv')
df = pd.read_csv(video_name+'_landmark.csv')
df.timeStamp = pd.to_datetime(df['timeStamp'].astype('str'))
# df.timeStamp = df.timeStamp.tz_convert("US/Eastern")

jointPoints = {
    'shoulder': {'a':'shoulder','b':'hip','c':'elbow'},
    'elbow': {'a':'elbow','b':'wrist','c':'shoulder'},
    'hip': {'a':'hip','b':'shoulder','c':'knee'},
    'knee': {'a':'knee','b':'hip','c':'ankle'}
    }

sides = ['l','r']
joints = ['shoulder','elbow','hip','knee']
axes = ['x','y','z']

for side in sides:
    print(side)
    for joint in joints:
        var = str(side+"_"+joint+"_angle")
        df[var] = df.apply(lambda x: getAngle(
            a = [x[str(side+"_"+jointPoints[joint]['a']+"_x")],
                 x[str(side+"_"+jointPoints[joint]['a']+"_y")],
                 x[str(side+"_"+jointPoints[joint]['a']+"_z")]], 
            b = [x[str(side+"_"+jointPoints[joint]['b']+"_x")],
                 x[str(side+"_"+jointPoints[joint]['b']+"_y")],
                 x[str(side+"_"+jointPoints[joint]['b']+"_z")]], 
            c = [x[str(side+"_"+jointPoints[joint]['c']+"_x")],
                 x[str(side+"_"+jointPoints[joint]['c']+"_y")],
                 x[str(side+"_"+jointPoints[joint]['c']+"_z")]]), 
            axis = 1
            )

df = df.set_index('timeStamp').resample('50ms').mean()

In [ ]:

# Get postural angles
# https://stackoverflow.com/questions/35176451/python-code-to-calculate-angle-between-three-point-using-their-3d-coordinates
def getAngle(a,b,c):
    
    # a is point of target
    # joint which you get
    # angle for

    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = b - c
    cosine_angle = np.dot(ba,bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

video_name = 'FV_1.mp4'
# df = pd.read_csv(video_name+'_world_landmark.csv')
df = pd.read_csv(video_name+'_landmark.csv')
df.head()

# df['l_shoulder_angle'] = df.apply(lambda x: getAngle(
#     a = [x.l_hip_x,x.l_hip_y,x.l_hip_z], 
#     b = [x.l_shoulder_x,x.l_shoulder_y,x.l_shoulder_z], 
#     c = [x.l_elbow_x,x.l_elbow_y,x.l_elbow_z]),
#     axis = 1
#     )
df['l_shoulder_angle'] = df.apply(lambda x: getAngle(
    a = [x.l_hip_x,x.l_hip_y], 
    b = [x.l_shoulder_x,x.l_shoulder_y], 
    c = [x.l_elbow_x,x.l_elbow_y]),
    axis = 1
    )
# df['r_shoulder_angle'] = df.apply(lambda x: getAngle(
#     a = [x.r_hip_x,x.r_hip_y,x.r_hip_z], 
#     b = [x.r_shoulder_x,x.r_shoulder_y,x.r_shoulder_z], 
#     c = [x.r_elbow_x,x.r_elbow_y,x.r_elbow_z]),
#     axis = 1
#     )

df['r_shoulder_angle'] = df.apply(lambda x: getAngle(
    a = [x.r_hip_x,x.r_hip_y], 
    b = [x.r_shoulder_x,x.r_shoulder_y], 
    c = [x.r_elbow_x,x.r_elbow_y]),
    axis = 1
    )

# df['l_elbow_angle'] = df.apply(lambda x: getAngle(
#     a = [x.l_shoulder_x,x.l_shoulder_y,x.l_shoulder_z], 
#     b = [x.l_elbow_x,x.l_elbow_y,x.l_elbow_z],
#     c = [x.l_wrist_x,x.l_wrist_y,x.l_wrist_z]),
#     axis = 1
#     )

df['l_elbow_angle'] = df.apply(lambda x: getAngle(
    # a = [x.l_shoulder_x,x.l_shoulder_y], 
    # b = [x.l_elbow_x,x.l_elbow_y],
    # c = [x.l_wrist_x,x.l_wrist_y]),
    a = [x.l_elbow_x,x.l_elbow_y], 
    b = [x.l_shoulder_x,x.l_shoulder_y],
    c = [x.l_wrist_x,x.l_wrist_y]),
    axis = 1
    )

# df['r_elbow_angle'] = df.apply(lambda x: getAngle(
#     a = [x.r_shoulder_x,x.r_shoulder_y,x.r_shoulder_z], 
#     b = [x.r_elbow_x,x.r_elbow_y,x.r_elbow_z],
#     c = [x.r_wrist_x,x.r_wrist_y,x.r_wrist_z]),
#     axis = 1
#     )

df['r_elbow_angle'] = df.apply(lambda x: getAngle(
    a = [x.r_shoulder_x,x.r_shoulder_y], 
    b = [x.r_elbow_x,x.r_elbow_y],
    c = [x.r_wrist_x,x.r_wrist_y]),
    axis = 1
    )

In [ ]:
# OLD Get Plot

print(len(df))
# For FV 1
#mask = (df.index >= (df.index.min() + pd.Timedelta(unit='second',value=8))) & (df.index <= (df.index.min() + pd.Timedelta(unit='second', value=32)))

# for SV 1
mask = (df.index >= (df.index.min() + pd.Timedelta(unit='second',value=18))) & (df.index <= (df.index.min() + pd.Timedelta(unit='second', value=42)))

df_trim = df[mask]
print(len(df_trim))

df_trim.plot.line(
    # for FV
    # y=['l_elbow_angle','r_elbow_angle',
    #    'l_shoulder_angle','r_shoulder_angle']
    # for SV
    y=['l_shoulder_angle','l_hip_angle','l_knee_angle']
    )

plt.ylim(0,180)
plt.title(video_name)
plt.savefig('SV_1.png')
plt.show()

In [ ]:
# OLD Working with video clips

videoFV = mpy.VideoFileClip("pose_FV_1.mp4")
videoFV = videoFV.subclipped("00:00:08","00:00:32")
chartFV = mpy.ImageClip('FV_1.png')#.set_start(0).set_duration(videoFV.duration)
chartFV.duration = videoFV.duration

videoSV = mpy.VideoFileClip("pose_SV_1.mp4")
videoSV = videoSV.subclipped("00:00:16","00:00:40")
chartSV = mpy.ImageClip('SV_1.png')
chartSV.duration = videoSV.duration

combined = mpy.clips_array(
    [[videoSV,chartSV],
    [videoFV,chartFV]]
)
# combined.preview(fps=30)
combined.write_videofile('CombV_1.mp4')

In [ ]:
# example for auomated ros bag to video NEEDS WORK
# https://www.google.com/search?client=firefox-b-1-d&q=python+bag+file+to+pngs
import rosbag
import cv_bridge
import os

def bag_to_images(bag_file, output_dir, image_topic):
    """Extract images from a rosbag."""
    bag = rosbag.Bag(bag_file, "r")
    bridge = cv_bridge.CvBridge()

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for topic, msg, t in bag.read_messages(topics=[image_topic]):
        try:
            cv_image = bridge.imgmsg_to_cv2(msg, desired_encoding="bgr8")
        except cv_bridge.CvBridgeError as e:
            print(e)

        image_name = f"{t.secs}_{t.nsecs}.png"
        image_path = os.path.join(output_dir, image_name)
        cv2.imwrite(image_path, cv_image)

    bag.close()

if __name__ == '__main__':
    bag_file = "your_bag_file.bag"  # Replace with your bag file path
    output_dir = "output_images"  # Replace with your desired output directory
    image_topic = "/camera/image_raw"  # Replace with the image topic in your bag file

    bag_to_images(bag_file, output_dir, image_topic)